In [1]:
# ==============================================================================
# PIPELINE 2: Notice-Level Matching & Bias Analysis
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.profiling import generate_minimal_report
from src.preparation.matching import split_by_notice, finalize_datasets
from src.preparation.match_analysis import compare_distributions, compare_metrics

# Global settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# ------------------------------------------------------------------------------
# STEP 1: Load Prepared Data from Pipeline 1
# ------------------------------------------------------------------------------
print("Loading notice-level datasets...")
df_cfc = pd.read_parquet("data/prepared/CFC_2018_2023.parquet")
df_can = pd.read_parquet("data/prepared/CAN_2018_2023.parquet")

Loading notice-level datasets...


In [2]:
# ------------------------------------------------------------------------------
# STEP 2: Notice-Level Split (Matches vs. Orphans)
# ------------------------------------------------------------------------------
split_results = split_by_notice(df_cfc, df_can)

cfc_matched = split_results['cfc_matched']
cfc_orphans = split_results['cfc_orphans']
can_matched = split_results['can_matched']
can_orphans = split_results['can_orphans']

 -> Notice-Level Match: 1,219,698 shared IDs identified.
    CFC: 6,309,001 matches | 1,378,448 orphans.
    CAN: 4,844,277 matches | 994,786 orphans.


In [3]:
# ------------------------------------------------------------------------------
# STEP 3: Bias Analysis (Censoring & Procedure Checks)
# ------------------------------------------------------------------------------

# 3.1 Temporal Distribution (Censoring Check)
print("Analyzing temporal distribution for CFC...")
cfc_time_analysis = compare_distributions(cfc_matched, cfc_orphans, feature='YEAR')
display(cfc_time_analysis)

print("\nAnalyzing temporal distribution for CAN...")
can_time_analysis = compare_distributions(can_matched, can_orphans, feature='YEAR')
display(can_time_analysis)

# 3.2 Procedure Type Distribution (Direct Award Check for CAN)
print("\nAnalyzing procedure type bias for CAN...")
can_proc_analysis = compare_distributions(can_matched, can_orphans, feature='TOP_TYPE', normalize=True)
display(can_proc_analysis.head(10))

Analyzing temporal distribution for CFC...
 -> Analyzed distribution bias for feature 'YEAR'.


,Matched,Orphans,Orphan_Rate (%)
YEAR,,,
2018,798161,195843,19.70
2019,1143788,183388,13.82
2020,1127370,161832,12.55
2021,1161568,180934,13.48
2022,1515603,203010,11.81
2023,562511,453441,44.63



Analyzing temporal distribution for CAN...
 -> Analyzed distribution bias for feature 'YEAR'.


,Matched,Orphans,Orphan_Rate (%)
YEAR,,,
2018,284429,404460,58.71
2019,702301,242572,25.67
2020,902138,123204,12.02
2021,988627,125115,11.23
2022,966108,51529,5.06
2023,1000674,47906,4.57



Analyzing procedure type bias for CAN...
 -> Analyzed distribution bias for feature 'TOP_TYPE'.


,Matched,Orphans,Difference (%-Points)
TOP_TYPE,,,
AWP,0.00,0.16,0.16
COD,0.08,0.11,0.03
INP,0.03,0.03,-0.00
NIC,3.88,6.03,2.15
NOC,0.10,12.26,12.16
OPE,93.30,74.59,-18.70
RES,2.61,6.82,4.20


In [4]:
# ------------------------------------------------------------------------------
# STEP 4: Deep Dive: Open Procedure (OPE) Analysis
# ------------------------------------------------------------------------------
# Filtering for Open Procedures to isolate systemic matching failures
m_ope_can = can_matched[can_matched['TOP_TYPE'] == 'OPE']
o_ope_can = can_orphans[can_orphans['TOP_TYPE'] == 'OPE']

# 4.1 Temporal distribution of OPE orphans
print("Temporal distribution of OPE orphans (CAN):")
display(compare_distributions(m_ope_can, o_ope_can, feature='YEAR', normalize=True))

# 4.2 Financial metrics comparison for OPE
print("\nFinancial metrics comparison for OPE (CAN):")
display(compare_metrics(m_ope_can, o_ope_can, metrics=['TOTAL_VALUE_EUR']))

Temporal distribution of OPE orphans (CAN):
 -> Analyzed distribution bias for feature 'YEAR'.


,Matched,Orphans,Difference (%-Points)
YEAR,,,
2018,5.97,46.72,40.75
2019,14.62,26.40,11.77
2020,18.80,10.99,-7.81
2021,20.48,11.67,-8.80
2022,19.74,2.33,-17.41
2023,20.39,1.90,-18.50



Financial metrics comparison for OPE (CAN):
 -> Compared central tendencies for 1 metric(s).


,Metric,Matched_Median,Orphan_Median,Matched_Mean,Orphan_Mean
0,TOTAL_VALUE_EUR,360000.0,186000.0,11692971.17,4787475.54


In [5]:
# ------------------------------------------------------------------------------
# STEP 5: Finalize & Save Checkpoints
# ------------------------------------------------------------------------------
# Discard orphans and keep only validated matches
df_cfc_final, df_can_final = finalize_datasets(split_results)

df_cfc_final.to_parquet("data/prepared/CFC_2018_2023.parquet", index=False)
df_can_final.to_parquet("data/prepared/CAN_2018_2023.parquet", index=False)

# Generate a minimal report for the full CFC data
generate_minimal_report(
    df=df_cfc_final,
    output_path="reports/processed/cfc_minimal_notices.html",
    title="CFC Notice Matching (minimal; full data)"
)

# Generate a minimal report for the full CAN data
generate_minimal_report(
    df=df_can_final,
    output_path="reports/processed/can_minimal_notices.html",
    title="CAN Notice Matching (minimal; full data)"
)

 -> Finalized datasets. Orphans discarded.
    CFC Final Rows: 6,309,001
    CAN Final Rows: 4,844,277
Generating minimal report: 'CFC Notice Matching (minimal; full data)' ...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:02<00:00, 10.62it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 31/31 [00:01<00:00, 16.97it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> Minimal report successfully saved to: reports/processed/can_minimal_notices.html

